# DependenSee: анализ статистики.

Это разбор конкретных историй с форумов, Stack Exchange, Reddit и GitHub. Для нормальной визуализации запускать файл сверху вниз. CSV файл необходим, без него не заработает

## TL:DR
- Больше всего материала дал Reddit, но и остальные площадки представлены заметно.
- Зависимости и очистка почти всегда идут вместе: 105 из 134 публикаций говорят об обоих сразу.
- Визуализация встречается редко: 8 публикаций из 134. Тем не менее, запрос на подобное присутствует, а сам я считаю GUI довольно удобным средством в анализе большого количества элементов (из других GUI приложений для анализа чего-то системного и связанного с графами можно вспомнить helvum - менеджер аудиоустройств в linux-системах)
- Среди аналогов нет одного инструмента, который закрывает весь спектр проблем, которые призван решить DependenSee

## 1. Баланс источников



In [ ]:
# Накидывание базы для работы всего ниже
from __future__ import annotations

import csv
from collections import Counter
from datetime import date
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import BoundaryNorm, LinearSegmentedColormap
from matplotlib.patches import Patch

HERE = Path.cwd()
INPUT = (HERE / ".." / "discussions.csv").resolve()
if not INPUT.exists():
    for parent in (HERE, *HERE.parents):
        candidate = parent / "research_data" / "discussions.csv"
        if candidate.exists():
            INPUT = candidate
            break
FIG_DIR = HERE / "figures"

TOTAL_EXPECTED = 134

CATEGORIES = [
    "removal_safety",
    "unused_packages",
    "cleanup_disk_space",
    "package_tool_usability",
    "dependency_opacity",
    "orphaned_packages",
    "visualization_request",
    "package_inventory",
    "remote_server_analysis",
]

CATEGORY_RU = {
    "removal_safety": "Безопасность удаления",
    "unused_packages": "Неиспользуемые пакеты",
    "cleanup_disk_space": "Очистка и место на диске",
    "package_tool_usability": "Удобство инструментов",
    "dependency_opacity": "Непрозрачность зависимостей",
    "orphaned_packages": "Orphan-пакеты",
    "visualization_request": "Запрос визуализации",
    "package_inventory": "Инвентаризация пакетов",
    "remote_server_analysis": "Удалённый сервер",
}

CATEGORY_SHORT = {
    "removal_safety": "Безопасность",
    "unused_packages": "Неиспользуемые",
    "cleanup_disk_space": "Очистка",
    "package_tool_usability": "Удобство",
    "dependency_opacity": "Непрозрачность",
    "orphaned_packages": "Orphan",
    "visualization_request": "Визуализация",
    "package_inventory": "Инвентаризация",
    "remote_server_analysis": "Сервер",
}

FLAGS = [
    "mentions_dependencies",
    "mentions_cleanup",
    "mentions_safety",
    "mentions_disk_space",
    "mentions_visualization",
]

FLAG_RU = {
    "mentions_dependencies": "Зависимости",
    "mentions_cleanup": "Очистка",
    "mentions_safety": "Безопасность",
    "mentions_disk_space": "Место на диске",
    "mentions_visualization": "Визуализация",
}

FLAG_SHORT = {
    "mentions_dependencies": "Зависимости",
    "mentions_cleanup": "Очистка",
    "mentions_safety": "Безопасность",
    "mentions_disk_space": "Место",
    "mentions_visualization": "Визуализация",
}

CATEGORY_COLORS = {
    "removal_safety": "#b03a2e",
    "unused_packages": "#d68910",
    "cleanup_disk_space": "#d4ac0d",
    "package_tool_usability": "#2e86c1",
    "dependency_opacity": "#7d3c98",
    "orphaned_packages": "#148f77",
    "visualization_request": "#27ae60",
    "package_inventory": "#7f8c8d",
    "remote_server_analysis": "#34495e",
}

FLAG_COLORS = {
    "mentions_dependencies": "#2e86c1",
    "mentions_cleanup": "#d68910",
    "mentions_safety": "#b03a2e",
    "mentions_disk_space": "#148f77",
    "mentions_visualization": "#7d3c98",
}

PLATFORM_ORDER = [
    "Reddit",
    "Ask Ubuntu",
    "GitHub",
    "Fedora Discussion",
    "Unix & Linux SE",
    "Arch Linux Forums",
]

PLATFORM_COLORS = {
    "Reddit": "#ff6b35",
    "Ask Ubuntu": "#e95420",
    "GitHub": "#24292f",
    "Fedora Discussion": "#3c6eb4",
    "Unix & Linux SE": "#0f7b6c",
    "Arch Linux Forums": "#1793d1",
}

DISTRO_FAMILIES = ["Debian/Ubuntu", "Fedora/RHEL", "Arch", "Не указан"]

DISTRO_FAMILY_RU = {
    "Debian/Ubuntu": "Debian / Ubuntu",
    "Fedora/RHEL": "Fedora / RHEL",
    "Arch": "Arch и производные",
    "Не указан": "Не указан",
}

DISTRO_TO_FAMILY = {
    "Ubuntu": "Debian/Ubuntu",
    "Debian": "Debian/Ubuntu",
    "Linux Mint": "Debian/Ubuntu",
    "Pop!_OS": "Debian/Ubuntu",
    "Lubuntu": "Debian/Ubuntu",
    "Kubuntu": "Debian/Ubuntu",
    "Xubuntu": "Debian/Ubuntu",
    "Ubuntu Studio": "Debian/Ubuntu",
    "Raspbian": "Debian/Ubuntu",
    "Fedora": "Fedora/RHEL",
    "CentOS": "Fedora/RHEL",
    "Arch Linux": "Arch",
    "Manjaro": "Arch",
    "CachyOS": "Arch",
}


def platform_of(source: str) -> str:
    if source.startswith("Reddit"):
        return "Reddit"
    if source.startswith("GitHub"):
        return "GitHub"
    if source == "Unix & Linux Stack Exchange":
        return "Unix & Linux SE"
    return source

def distro_family_of(distro: str) -> str:
    if not distro:
        return "Не указан"
    return DISTRO_TO_FAMILY.get(distro, distro)

def pct(part: float, total: float) -> float:
    return round(100.0 * part / total, 1) if total else 0.0

def load_rows() -> list[dict]:
    with INPUT.open(encoding="utf-8", newline="") as handle:
        rows = list(csv.DictReader(handle))

    for row in rows:
        row["date_obj"] = date.fromisoformat(row["date"])
        row["year"] = row["date_obj"].year
        row["platform"] = platform_of(row["source"])
        row["distro_family"] = distro_family_of(row["distro"])
        row["score_num"] = int(row["score"]) if row["score"].strip() else None
        row["comments_num"] = (
            int(row["comments_count"]) if row["comments_count"].strip() else None
        )
        for flag in FLAGS:
            row[flag] = row[flag].strip().lower() == "true"

    if len(rows) != TOTAL_EXPECTED:
        raise SystemExit(
            f"Expected {TOTAL_EXPECTED} rows in {INPUT}, got {len(rows)}. "
            "Update the analysis or the input deliberately."
        )
    return rows

def style_axes(ax: plt.Axes) -> None:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color("#b0b0b0")
    ax.spines["bottom"].set_color("#b0b0b0")
    ax.tick_params(colors="#333333", labelsize=9)
    ax.grid(axis="x", color="#e6e6e6", linewidth=0.8)
    ax.set_axisbelow(True)

def hbar(
    ax: plt.Axes,
    labels: list[str],
    values: list[float],
    colors,
    total: float | None = None,
    value_fmt: str = "{v:g} ({p:.1f}%)",
    xlabel: str = "Публикаций",
) -> None:
    y = np.arange(len(labels))
    bars = ax.barh(
        y,
        values,
        color=colors,
        height=0.68,
        edgecolor="white",
        linewidth=0.6,
    )
    ax.set_yticks(y)
    ax.set_yticklabels(labels)
    ax.invert_yaxis()
    max_value = max(values) if values else 1
    for bar, value in zip(bars, values):
        if total is not None:
            label = value_fmt.format(v=value, p=pct(value, total))
        else:
            label = value_fmt.format(v=value, p=0)
        ax.text(
            value + max_value * 0.015,
            bar.get_y() + bar.get_height() / 2,
            label,
            va="center",
            fontsize=9,
            color="#222222",
        )
    ax.set_xlim(0, max_value * 1.22)
    ax.set_xlabel(xlabel, fontsize=9, color="#444444")
    style_axes(ax)


def finish(
    fig: plt.Figure,
    ax: plt.Axes,
    path: Path,
    title: str,
    subtitle: str = "",
) -> None:
    """Показать график прямо в ноутбуке (path сохранён па приколу)."""
    ax.set_title(title, loc="left", fontsize=15, fontweight="bold", pad=24)
    if subtitle:
        ax.text(
            0.0,
            1.035,
            subtitle,
            transform=ax.transAxes,
            fontsize=9,
            color="#5d6d7e",
            va="bottom",
            ha="left",
        )
    plt.show()


rows = load_rows()


In [ ]:
def figure_platform_distribution(rows: list[dict]) -> None:
    total = len(rows)
    counts = Counter(row["platform"] for row in rows)
    labels = PLATFORM_ORDER
    values = [counts.get(label, 0) for label in labels]
    colors = [PLATFORM_COLORS[label] for label in labels]

    fig, ax = plt.subplots(figsize=(9.2, 4.8))
    hbar(ax, labels, values, colors, total=total)
    finish(
        fig,
        ax,
        FIG_DIR / "01_platform_distribution.png",
        "Публикации по площадкам",
        f"N = {total}; доли считаются от всего корпуса",
    )

figure_platform_distribution(rows)



Reddit дал 46 публикаций, почти треть всего материала. Дальше идут Ask Ubuntu (25), GitHub (20), Fedora Discussion (20), Unix & Linux Stack Exchange (17) и Arch Linux Forums (6).

Внутри Reddit основная масса пришла из `r/linuxquestions` (29) и `r/linux4noobs` (15). То есть это в основном вопросы новичков и людей, которые столкнулись с конкретной проблемой, а не теоретические споры.



In [ ]:
def figure_distro_distribution(rows: list[dict]) -> None:
    total = len(rows)
    counts = Counter(row["distro"] or "Не указан" for row in rows).most_common()
    labels = [item[0] for item in counts]
    values = [item[1] for item in counts]
    colors = []
    for label in labels:
        family = distro_family_of("" if label == "Не указан" else label)
        colors.append(
            {
                "Debian/Ubuntu": "#e67e22",
                "Fedora/RHEL": "#2e86c1",
                "Arch": "#1793d1",
                "Не указан": "#95a5a6",
            }.get(family, "#7f8c8d")
        )

    fig, ax = plt.subplots(figsize=(10.2, 6.0))
    hbar(ax, labels, values, colors, total=total)
    finish(
        fig,
        ax,
        FIG_DIR / "03_distro_distribution.png",
        "Дистрибутивы, указанные в публикациях",
        f"N = {total}; «Не указан» - 21 публикация без однозначного дистрибутива",
    )

figure_distro_distribution(rows)



По дистрибутивам ожидаемо лидируют Ubuntu (35) и Fedora (29). Дальше Arch Linux (17), Debian (9), Linux Mint (6). Ещё 21 публикация без явного дистрибутива.

Если собрать родственные дистрибутивы в семьи, получается так:

- Debian и Ubuntu: 60 публикаций;
- Fedora и RHEL: 31;
- Arch и производные: 22;
- дистрибутив не указан: 21.

Для проекта это полезное напоминание: большая часть болей приходит из мира APT и DNF, но Arch тоже заметен, особенно в теме orphan-пакетов (закономерная проблема модели rolling-release)

## 2. Проблемы



In [ ]:
def figure_category_distribution(rows: list[dict]) -> None:
    total = len(rows)
    counts = Counter(row["problem_category"] for row in rows).most_common()
    labels = [CATEGORY_RU[item[0]] for item in counts]
    values = [item[1] for item in counts]
    colors = [CATEGORY_COLORS[item[0]] for item in counts]

    fig, ax = plt.subplots(figsize=(10.2, 5.8))
    hbar(ax, labels, values, colors, total=total)
    finish(
        fig,
        ax,
        FIG_DIR / "04_category_distribution.png",
        "Основная проблема публикации",
        f"N = {total}; одна категория на публикацию",
    )

figure_category_distribution(rows)



Самая частая категория - безопасность удаления: 37 публикаций (27,6%). Люди боятся выполнить `autoremove` "лишних" пакетов и остаться без загрузки или без рабочего стола.

Дальше идут:

- неиспользуемые пакеты: 22 (16,4%);
- очистка и место на диске: 20 (14,9%);
- удобство инструментов: 16 (11,9%);
- непрозрачность зависимостей: 15 (11,2%);
- orphan-пакеты: 10 (7,5%);
- запрос визуализации: 7 (5,2%);
- инвентаризация: 4 (3,0%);
- удалённый сервер: 3 (2,2%).

Три первые категории дают 79 публикаций, то есть 59% корпуса. Если объяснить одной фразой: людям мешает не отсутствие инструмента удаление, а страх перед тем, что инструмент поведёт себя непредсказуемо (именно поэтому DependenSee не сосредоточен на удалении, а призван помочь самостоятельно проанализировать и удалить пакеты)



In [ ]:
def figure_yearly_dynamics(rows: list[dict]) -> None:
    years = sorted(set(row["year"] for row in rows))
    category_order = [
        category
        for category, _ in Counter(
            row["problem_category"] for row in rows
        ).most_common()
    ]
    category_counts = Counter(
        (row["year"], row["problem_category"]) for row in rows
    )
    year_totals = Counter(row["year"] for row in rows)

    fig, axes = plt.subplots(
        1, 2, figsize=(13.6, 6.2), gridspec_kw={"width_ratios": [1.05, 1]}
    )

    bottoms = np.zeros(len(years))
    for category in category_order:
        values = np.array(
            [category_counts.get((year, category), 0) for year in years],
            dtype=float,
        )
        axes[0].bar(
            years,
            values,
            bottom=bottoms,
            width=0.72,
            color=CATEGORY_COLORS[category],
            edgecolor="white",
            linewidth=0.5,
            label=CATEGORY_RU[category],
        )
        bottoms += values

    for year, total in zip(years, bottoms):
        axes[0].text(
            year,
            total + 0.5,
            f"{int(total)}",
            ha="center",
            va="bottom",
            fontsize=9,
            color="#333333",
        )

    axes[0].set_xticks(years)
    axes[0].set_ylim(0, max(bottoms) * 1.16)
    axes[0].set_ylabel("Публикаций", fontsize=9)
    axes[0].set_xlabel("Год публикации", fontsize=9)
    axes[0].set_title("Абсолютное число", loc="left", fontsize=11, fontweight="bold")
    axes[0].grid(axis="y", color="#e6e6e6", linewidth=0.8)
    axes[0].set_axisbelow(True)
    axes[0].spines["top"].set_visible(False)
    axes[0].spines["right"].set_visible(False)

    year_shares = np.array([year_totals[year] for year in years], dtype=float)
    bottoms = np.zeros(len(years))
    for category in category_order:
        values = np.array(
            [category_counts.get((year, category), 0) for year in years],
            dtype=float,
        )
        shares = np.divide(
            values,
            year_shares,
            out=np.zeros_like(values),
            where=year_shares > 0,
        ) * 100
        axes[1].bar(
            years,
            shares,
            bottom=bottoms,
            width=0.72,
            color=CATEGORY_COLORS[category],
            edgecolor="white",
            linewidth=0.5,
        )
        for x, share, value, bottom in zip(years, shares, values, bottoms):
            if share >= 12:
                axes[1].text(
                    x,
                    bottom + share / 2,
                    f"{int(value)}",
                    ha="center",
                    va="center",
                    fontsize=8,
                    color="white" if category not in {"cleanup_disk_space"} else "#333333",
                )
        bottoms += shares

    axes[1].set_xticks(years)
    axes[1].set_ylim(0, 100)
    axes[1].set_ylabel("Доля внутри года, %", fontsize=9)
    axes[1].set_xlabel("Год публикации", fontsize=9)
    axes[1].set_title("Состав внутри года", loc="left", fontsize=11, fontweight="bold")
    axes[1].grid(axis="y", color="#e6e6e6", linewidth=0.8)
    axes[1].set_axisbelow(True)
    axes[1].spines["top"].set_visible(False)
    axes[1].spines["right"].set_visible(False)

    handles = [
        Patch(facecolor=CATEGORY_COLORS[category], label=CATEGORY_RU[category])
        for category in category_order
    ]
    fig.legend(
        handles=handles,
        loc="lower center",
        ncol=3,
        frameon=False,
        fontsize=9,
        bbox_to_anchor=(0.5, -0.035),
    )
    fig.suptitle(
        "Динамика корпуса по годам",
        x=0.012,
        y=1.015,
        ha="left",
        fontsize=15,
        fontweight="bold",
    )
    fig.text(
        0.012,
        0.965,
        "N = 134; 2026 год неполный - публикации по 2 ноября",
        ha="left",
        fontsize=9,
        color="#5d6d7e",
    )
    fig.subplots_adjust(top=0.86, bottom=0.20, wspace=0.22)
    plt.show()


figure_yearly_dynamics(rows)



Не смотря на неравномерность, график показывает стабильные всплески интереса к проблеме.

## 3. Анализ популярных проблем



In [ ]:
def figure_flag_prevalence(rows: list[dict]) -> None:
    total = len(rows)
    counts = Counter(flag for row in rows for flag in FLAGS if row[flag])
    labels = [FLAG_RU[flag] for flag in FLAGS]
    values = [counts.get(flag, 0) for flag in FLAGS]
    colors = [FLAG_COLORS[flag] for flag in FLAGS]

    fig, ax = plt.subplots(figsize=(9.4, 4.5))
    hbar(ax, labels, values, colors, total=total)
    finish(
        fig,
        ax,
        FIG_DIR / "08_flag_prevalence.png",
        "Распространённость флагов",
        f"N = {total}; флаги не взаимоисключающие, поэтому сумма больше 100%",
    )

figure_flag_prevalence(rows)



Флаги - это темы, которые встречаются в тексте. Одна публикация может задевать несколько тем.

- Зависимости: 118 из 134 (88,1%).
- Очистка: 120 из 134 (89,6%).
- Безопасность: 57 из 134 (42,5%).
- Место на диске: 37 из 134 (27,6%).
- Визуализация: 8 из 134 (6,0%).

Зависимости и очистка пересекаются у 105 публикаций (78,4%). Это логично. Люди часто хотят удалить что-либо "лишнее", но боятся, что бесполезный на первый взгляд пакет повлечёт за собой каскадное разрушение всей системы.

Безопасность - вторая по важности тема после самих зависимостей и очистки. 20 из 57 публикаций с флагом безопасности находятся вне категории "безопасность удаления". То есть страх сломать систему встречается и в темах про неиспользуемые пакеты, удобство инструментов и непрозрачность зависимостей.



In [ ]:
def figure_category_flag_heatmap(rows: list[dict]) -> None:
    category_counts = Counter(row["problem_category"] for row in rows)
    category_flag = Counter(
        (row["problem_category"], flag)
        for row in rows
        for flag in FLAGS
        if row[flag]
    )
    categories = [
        category for category, _ in category_counts.most_common()
    ]
    matrix = np.array(
        [
            [
                pct(category_flag.get((category, flag), 0), category_counts[category])
                for flag in FLAGS
            ]
            for category in categories
        ]
    )

    fig, ax = plt.subplots(figsize=(9.8, 6.2))
    im = ax.imshow(matrix, cmap="Blues", aspect="auto", vmin=0, vmax=100)
    ax.set_xticks(np.arange(len(FLAGS)))
    ax.set_xticklabels([FLAG_SHORT[flag] for flag in FLAGS], rotation=28, ha="right")
    ax.set_yticks(np.arange(len(categories)))
    ax.set_yticklabels(
        [f"{CATEGORY_RU[category]} ({category_counts[category]})" for category in categories],
        fontsize=9,
    )
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            value = matrix[i, j]
            ax.text(
                j,
                i,
                f"{value:.0f}%",
                ha="center",
                va="center",
                fontsize=8.5,
                color="white" if value > 55 else "#1b2631",
            )
    ax.set_xticks(np.arange(-0.5, len(FLAGS), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, len(categories), 1), minor=True)
    ax.grid(which="minor", color="white", linewidth=1.4)
    ax.tick_params(which="minor", bottom=False, left=False)
    ax.tick_params(axis="both", length=0)
    cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
    cbar.set_label("Доля внутри категории, %", fontsize=8)
    cbar.ax.tick_params(labelsize=8)
    finish(
        fig,
        ax,
        FIG_DIR / "09_category_flag_heatmap.png",
        "Категория × флаг",
        "Значение - доля публикаций категории с данным флагом; в скобках - размер категории",
    )

figure_category_flag_heatmap(rows)



Эта таблица показывает, какие темы встречаются внутри каждой категории.

- В категории "безопасность удаления" 100% публикаций говорят о безопасности, 97% - об очистке, 19% - о месте на диске.
- В категории "очистка и место" 100% говорят о месте на диске и 0% - о безопасности. Тем не менее, я не исключаю, что это ошибка выбора формулировок, ведь людям интересны как минимум "Зависимости"
- В категории "запрос визуализации" 100% говорят о зависимостях и 0% - о безопасности (возможно так же из-за формулировок)
- В категории "orphan-пакеты" 100% говорят о зависимостях и очистке, но только 20% - о безопасности.



In [ ]:
def figure_platform_category_heatmap(rows: list[dict]) -> None:
    platform_counts = Counter(row["platform"] for row in rows)
    platform_category = Counter(
        (row["platform"], row["problem_category"]) for row in rows
    )
    matrix = np.array(
        [
            [platform_category.get((platform, category), 0) for category in CATEGORIES]
            for platform in PLATFORM_ORDER
        ]
    )

    fig, ax = plt.subplots(figsize=(12.2, 5.6))
    im = ax.imshow(matrix, cmap="Blues", aspect="auto", vmin=0, vmax=matrix.max())
    ax.set_xticks(np.arange(len(CATEGORIES)))
    ax.set_xticklabels(
        [CATEGORY_SHORT[category] for category in CATEGORIES],
        rotation=32,
        ha="right",
        fontsize=9,
    )
    ax.set_yticks(np.arange(len(PLATFORM_ORDER)))
    ax.set_yticklabels(
        [
            f"{platform} ({platform_counts.get(platform, 0)})"
            for platform in PLATFORM_ORDER
        ],
        fontsize=9,
    )
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            value = int(matrix[i, j])
            ax.text(
                j,
                i,
                str(value),
                ha="center",
                va="center",
                fontsize=9,
                color="white" if value > matrix.max() * 0.55 else "#1b2631",
            )
    ax.set_xticks(np.arange(-0.5, len(CATEGORIES), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, len(PLATFORM_ORDER), 1), minor=True)
    ax.grid(which="minor", color="white", linewidth=1.4)
    ax.tick_params(which="minor", bottom=False, left=False)
    ax.tick_params(axis="both", length=0)
    cbar = fig.colorbar(im, ax=ax, fraction=0.028, pad=0.02)
    cbar.set_label("Публикаций", fontsize=8)
    cbar.ax.tick_params(labelsize=8)
    finish(
        fig,
        ax,
        FIG_DIR / "06_platform_category_heatmap.png",
        "Площадка × категория",
        "Число публикаций; в скобках у площадки - её размер. Ячейки не нормированы.",
    )

figure_platform_category_heatmap(rows)



На этой карте видно, о чём чаще говорят на разных площадках.

- Ask Ubuntu заметно чаще про безопасность удаления: 12 из 25 публикаций.
- Unix & Linux Stack Exchange - про непрозрачность зависимостей: 6 из 17.
- Arch-семейство - про orphan-пакеты: 8 из 22.
- GitHub распределён ровнее: неиспользуемые пакеты, очистка, безопасность и orphan идут рядом.

Это состав моей подборки, а не рейтинг площадок. Но для понимания аудитории полезно: новички на Ask Ubuntu больше говорят о страхе удаления. Среди людей более углубленных чаще возникает желание узнать о том, почему и что стоит в их системе.

## 4. Визуализация



In [ ]:
def figure_visualization_gap(rows: list[dict]) -> None:
    visual = [row for row in rows if row["mentions_visualization"]]
    category_counts = Counter(row["problem_category"] for row in visual)
    categories = [category for category, _ in category_counts.most_common()]
    labels_left = [CATEGORY_RU[category] for category in categories]
    values_left = [category_counts[category] for category in categories]
    colors_left = [CATEGORY_COLORS[category] for category in categories]
    flags = [
        "mentions_dependencies",
        "mentions_cleanup",
        "mentions_disk_space",
        "mentions_safety",
    ]
    values_right = [
        sum(1 for row in visual if row[flag]) for flag in flags
    ]
    colors_right = [FLAG_COLORS[flag] for flag in flags]
    labels_right = [FLAG_RU[flag] for flag in flags]

    fig, axes = plt.subplots(
        1, 2, figsize=(12.8, 4.9), gridspec_kw={"width_ratios": [1, 1.05]}
    )
    hbar(
        axes[0],
        labels_left,
        values_left,
        colors_left,
        total=len(visual),
        xlabel="Публикаций",
    )
    axes[0].set_title(
        "Категория публикаций с упоминанием визуализации",
        loc="left",
        fontsize=11,
        fontweight="bold",
    )
    hbar(
        axes[1],
        labels_right,
        values_right,
        colors_right,
        total=len(visual),
        xlabel="Публикаций",
    )
    axes[1].set_title(
        "Какие флаги есть у этих публикаций",
        loc="left",
        fontsize=11,
        fontweight="bold",
    )
    fig.suptitle(
        "Визуализация зависимостей: 8 из 134 публикаций (6,0%)",
        x=0.012,
        y=1.035,
        ha="left",
        fontsize=15,
        fontweight="bold",
    )
    fig.text(
        0.012,
        0.975,
        "7 из 8 — прямая категория visualization_request, 1 - unused_packages (DS-009). Только 1 из 8 упоминает безопасность.",
        ha="left",
        fontsize=9,
        color="#5d6d7e",
    )
    fig.subplots_adjust(top=0.83, wspace=0.45)
    plt.show()


figure_visualization_gap(rows)



Визуализация - маленькая часть корпуса: 8 публикаций из 134 (6,0%). Семь из них - прямой запрос графа или дерева, ещё одна - DS-009, где визуализация упоминается внутри разговора об очистке.

Что важно: только одна из восьми публикаций вообще упоминает безопасность. Среди семи прямых запросов визуализации таких нет ни одной.

Вот что люди просят:

- DS-007: интерактивное дерево, которое объясняет, почему `dnf5` тянет неожиданные пакеты.
- DS-043: дерево обратных зависимостей вместо плоского списка.
- DS-044: дерево неустановленных зависимостей вместе с размерами файлов.
- DS-082: дерево разрешённых зависимостей APT.
- DS-058: аналог Dependency Walker для Linux, с рекурсивными библиотеками и импортируемыми функциями.
- DS-108: дерево зависимостей для набора пакетов как единого целого.
- DS-124: разбор раздутой установки перед очисткой.
- DS-009: инвентаризация и дерево зависимостей перед очисткой, с упоминанием безопасности.

Главная мысль: визуализация здесь нужна не для красоты и не для кнопки "удалить". Она нужна, чтобы ответить на вопрос "откуда этот пакет взялся и что он держит". Визуализация видится этим людям как самый простой способ объять систему зависимостей взглядом.

## 5. Вовлечённость



In [ ]:
def figure_comments(rows: list[dict]) -> None:
    data = []
    labels = []
    for platform in PLATFORM_ORDER:
        values = [
            row["comments_num"]
            for row in rows
            if row["platform"] == platform and row["comments_num"] is not None
        ]
        data.append(values)
        labels.append(f"{platform}\n(n={len(values)})")

    fig, ax = plt.subplots(figsize=(10.4, 5.8))
    positions = np.arange(len(data))
    box = ax.boxplot(
        data,
        positions=positions,
        widths=0.52,
        patch_artist=True,
        showfliers=False,
        medianprops={"color": "#111111", "linewidth": 1.8},
        whiskerprops={"color": "#7f8c8d", "linewidth": 1.0},
        capprops={"color": "#7f8c8d", "linewidth": 1.0},
    )
    for patch, platform in zip(box["boxes"], PLATFORM_ORDER):
        color = PLATFORM_COLORS[platform]
        patch.set_facecolor(color)
        patch.set_alpha(0.28)
        patch.set_edgecolor(color)
        patch.set_linewidth(1.2)

    rng = np.random.default_rng(42)
    for index, (platform, values) in enumerate(zip(PLATFORM_ORDER, data)):
        jitter = rng.uniform(-0.15, 0.15, len(values))
        ax.scatter(
            index + jitter,
            values,
            s=14,
            color=PLATFORM_COLORS[platform],
            alpha=0.55,
            edgecolor="none",
            zorder=3,
        )
        median = float(np.median(values))
        ax.text(
            index + 0.30,
            median,
            f"мед. {median:g}",
            ha="left",
            va="center",
            fontsize=8.5,
            color="#333333",
        )

    ax.set_xticks(positions)
    ax.set_xticklabels(labels, fontsize=8.5)
    ax.set_ylabel("Комментарии / ответы / реплики", fontsize=9)
    ax.set_ylim(0, max(max(values) for values in data) * 1.15)
    ax.grid(axis="y", color="#e6e6e6", linewidth=0.8)
    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    finish(
        fig,
        ax,
        FIG_DIR / "11_comments_by_platform.png",
        "Вовлечённость: комментарии по площадкам",
        "Ящики — квартили, точки - публикации; для Stack Exchange это число ответов, а не комментариев",
    )

figure_comments(rows)



На Reddit типичная публикация собирает 8 комментариев, на Fedora - 7,5, на Arch - 5,5. На GitHub, Ask Ubuntu и Unix & Linux медиана равна 1.

Но сравнивать эти числа напрямую нельзя. На Stack Exchange поле означает число ответов, а не комментариев. На Reddit это комментарии, на Fedora - ответы в теме, на GitHub - комментарии к issue. У площадок разная механика общения.

По рейтингам картина ещё проще: медиана равна 1 на всех трёх площадках, где рейтинг вообще есть. Средние значения вытягивают отдельные выбросы: на Reddit есть пост с 81 баллом, на Unix & Linux - с 22.

Самые обсуждаемые истории:

- 79 комментариев: стратегия против накопления неиспользуемых пакетов;
- 62 комментария: "что это значит?" после `autoremove`;
- 60 комментариев: `autoremove` сломал загрузку;
- 40 комментариев: bash-скрипт для обновления и очистки;
- 37 комментариев: `dnf autoremove` хочет удалить Qt.

## 6. Что уже есть на рынке



In [ ]:
ANALOG_FEATURES = [
    ("graph", "Граф"),
    ("whole_system", "Вся система"),
    ("why_chain", "Why-цепочки"),
    ("multi_manager", "Мульти-менеджер"),
    ("orphan_detection", "Orphan"),
    ("removal_preview", "Предпросмотр\nудаления"),
    ("export", "Экспорт"),
    ("disk_size", "Размеры"),
    ("headless", "Headless/CLI"),
    ("install_modeling", "Моделирование\nустановки"),
]

ANALOGS = [
    {
        "tool": "LDG (Linux Dependency Guide)",
        "short": "LDG",
        "class": "Прямой аналог",
        "url": "https://github.com/RubensRainelli/linux-dependency-guide",
        "values": {
            "graph": 1.0,
            "whole_system": 1.0,
            "why_chain": 0.5,
            "multi_manager": 1.0,
            "orphan_detection": 0.0,
            "removal_preview": 0.0,
            "export": 1.0,
            "disk_size": 0.0,
            "headless": 1.0,
            "install_modeling": 0.0,
        },
        "note": (
            "Интерактивный HTML-граф, JSON-инвентаризация и сравнение экспортов; "
            "перемещение по цепочкам есть, отдельной why-команды нет. Поиск orphan, "
            "размеры пакетов и предпросмотр удаления в просмотренной документации не заявлены."
        ),
    },
    {
        "tool": "Graphite",
        "short": "Graphite",
        "class": "Прямой аналог",
        "url": "https://github.com/CacheUseOnly/graphite",
        "values": {
            "graph": 1.0,
            "whole_system": 1.0,
            "why_chain": 0.5,
            "multi_manager": 0.0,
            "orphan_detection": 0.0,
            "removal_preview": 0.0,
            "export": 0.0,
            "disk_size": 0.0,
            "headless": 0.0,
            "install_modeling": 0.0,
        },
        "note": (
            "Интерактивный направленный граф установленной APT-системы и подсветка "
            "подграфа выбранного пакета; экспорт, orphan-поиск и удаление не заявлены."
        ),
    },
    {
        "tool": "Starfish",
        "short": "Starfish",
        "class": "Прямой аналог",
        "url": "https://github.com/Seafoam-Labs/Starfish",
        "values": {
            "graph": 1.0,
            "whole_system": 0.5,
            "why_chain": 0.5,
            "multi_manager": 0.0,
            "orphan_detection": 0.0,
            "removal_preview": 0.0,
            "export": 0.0,
            "disk_size": 0.0,
            "headless": 0.0,
            "install_modeling": 0.0,
        },
        "note": (
            "Интерактивный граф прямых зависимостей и Required By для Arch/ALPM; "
            "общесистемная карта, экспорт и сценарий удаления не подтверждены."
        ),
    },
    {
        "tool": "debtree",
        "short": "debtree",
        "class": "Специализированный инструмент",
        "url": "https://manpages.debian.org/trixie/debtree/debtree.1.en.html",
        "values": {
            "graph": 1.0,
            "whole_system": 0.0,
            "why_chain": 1.0,
            "multi_manager": 0.0,
            "orphan_detection": 0.0,
            "removal_preview": 0.0,
            "export": 0.5,
            "disk_size": 0.0,
            "headless": 1.0,
            "install_modeling": 0.0,
        },
        "note": (
            "Генерирует DOT для выбранного Debian-пакета: прямые, обратные и "
            "build-зависимости; визуализация требует Graphviz. Это не карта всей системы."
        ),
    },
    {
        "tool": "apt-rdepends",
        "short": "apt-rdepends",
        "class": "Специализированный инструмент",
        "url": "https://manpages.debian.org/bookworm/apt-rdepends/apt-rdepends.1.en.html",
        "values": {
            "graph": 0.5,
            "whole_system": 0.0,
            "why_chain": 1.0,
            "multi_manager": 0.0,
            "orphan_detection": 0.0,
            "removal_preview": 0.0,
            "export": 0.5,
            "disk_size": 0.0,
            "headless": 1.0,
            "install_modeling": 0.0,
        },
        "note": (
            "Рекурсивные прямые и обратные зависимости, DOT-подобный вывод; "
            "нет GUI, ограничения по глубине и виртуальным пакетам."
        ),
    },
    {
        "tool": "pacwall",
        "short": "pacwall",
        "class": "Специализированный инструмент",
        "url": "https://github.com/Kharacternyk/pacwall",
        "values": {
            "graph": 1.0,
            "whole_system": 1.0,
            "why_chain": 0.0,
            "multi_manager": 0.0,
            "orphan_detection": 1.0,
            "removal_preview": 0.0,
            "export": 0.5,
            "disk_size": 1.0,
            "headless": 0.0,
            "install_modeling": 0.0,
        },
        "note": (
            "Граф-обои всей pacman-системы: площадь узла связана с размером, "
            "подсвечены orphan/unneeded. Не интерактивен, не управляет удалением, pacman-only."
        ),
    },
    {
        "tool": "mpm (Meta Package Manager)",
        "short": "mpm",
        "class": "Смежное решение",
        "url": "https://github.com/kdeldycke/meta-package-manager",
        "values": {
            "graph": 0.0,
            "whole_system": 0.0,
            "why_chain": 0.0,
            "multi_manager": 1.0,
            "orphan_detection": 1.0,
            "removal_preview": 0.0,
            "export": 1.0,
            "disk_size": 0.0,
            "headless": 1.0,
            "install_modeling": 0.0,
        },
        "note": (
            "Мультименеджерный CLI: inventory, dump/restore, список orphan-пакетов, "
            "SBOM/JSON/TOML. Графа зависимостей и предпросмотра удаления нет; "
            "отчёт об изменениях появляется после выполнения команды."
        ),
    },
    {
        "tool": "aptitude",
        "short": "aptitude",
        "class": "Смежное решение",
        "url": "https://www.debian.org/doc/manuals/aptitude/rn01re01.en.html",
        "values": {
            "graph": 0.0,
            "whole_system": 0.0,
            "why_chain": 1.0,
            "multi_manager": 0.0,
            "orphan_detection": 1.0,
            "removal_preview": 1.0,
            "export": 0.5,
            "disk_size": 0.5,
            "headless": 1.0,
            "install_modeling": 1.0,
        },
        "note": (
            "TUI/CLI с why/why-not, автоматически установленными пакетами и "
            "предпросмотром планируемых действий, включая установку и размеры. "
            "Не строит визуальный граф и работает только с APT."
        ),
    },
]

In [ ]:
def figure_analog_heatmap() -> None:
    matrix = np.array(
        [[item["values"][key] for key, _ in ANALOG_FEATURES] for item in ANALOGS]
    )
    labels = [
        f"{item['short']} ({item['class'].split()[0].lower()})" for item in ANALOGS
    ]

    cmap = LinearSegmentedColormap.from_list(
        "feature_scale", ["#f4f6f7", "#aed6f1", "#1f618d"]
    )
    norm = BoundaryNorm([-0.25, 0.25, 0.75, 1.25], cmap.N)

    fig, ax = plt.subplots(figsize=(14.4, 7.0))
    im = ax.imshow(matrix, cmap=cmap, norm=norm, aspect="auto")
    ax.set_xticks(np.arange(len(ANALOG_FEATURES)))
    ax.set_xticklabels(
        [label for _, label in ANALOG_FEATURES], rotation=30, ha="right", fontsize=9
    )
    ax.set_yticks(np.arange(len(ANALOGS)))
    ax.set_yticklabels(labels, fontsize=9)
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            value = matrix[i, j]
            text = "да" if value == 1.0 else ("част." if value == 0.5 else "нет")
            ax.text(
                j,
                i,
                text,
                ha="center",
                va="center",
                fontsize=8.5,
                color="white" if value == 1.0 else "#1b2631",
            )
    for boundary in (2.5, 5.5):
        ax.axhline(boundary, color="white", linewidth=2.2)
    ax.set_xticks(np.arange(-0.5, len(ANALOG_FEATURES), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, len(ANALOGS), 1), minor=True)
    ax.grid(which="minor", color="white", linewidth=1.2)
    ax.tick_params(which="minor", bottom=False, left=False)
    ax.tick_params(axis="both", length=0)
    cbar = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02, ticks=[0, 0.5, 1])
    cbar.ax.set_yticklabels(["нет", "частично", "да"], fontsize=8)
    cbar.set_label("Подтверждено документацией", fontsize=8)
    finish(
        fig,
        ax,
        FIG_DIR / "14_analog_feature_heatmap.png",
        "Матрица подтверждённых функций аналогов",
        "8 инструментов; 0 - не заявлено/не подтверждено, 0,5 - частично, 1 - явно заявлено. URPM исключён (HTTP 404).",
    )

figure_analog_heatmap()



Я сравнил 8 инструментов: LDG, Graphite, Starfish, debtree, apt-rdepends, pacwall, mpm и aptitude. Смотрел документацию и README. 

Коротко по инструментам:

- LDG: мульти-менеджерный аудит, интерактивный HTML-граф, экспорт и сравнение инвентаризаций. Ближе всего к идее "увидеть всю систему".
- Graphite: красивый интерактивный граф установленной APT-системы. Но только APT и без orphan-поиска.
- Starfish: интерактивный граф для Arch, с прямыми и обратными зависимостями.
- debtree: DOT-граф для выбранного Debian-пакета. Мощно, но это CLI и не карта всей системы.
- apt-rdepends: рекурсивные зависимости и обратные зависимости в консоли.
- pacwall: граф-обои всей pacman-системы. Показывает размеры и orphan-пакеты, но это статичная картинка.
- mpm: мульти-менеджерный inventory, снимки, восстановление, список orphan-пакетов и экспорт. Графа зависимостей нет.
- aptitude: why/why-not, автоматически установленные пакеты, предпросмотр действий и моделирование установки. Но только APT и без визуального графа.



In [ ]:
def figure_analog_coverage() -> None:
    ordered = sorted(
        ANALOGS, key=lambda item: sum(item["values"].values()), reverse=True
    )
    labels = [
        f"{item['short']} — {item['class'].split()[0].lower()}" for item in ordered
    ]
    values = [sum(item["values"].values()) for item in ordered]
    colors = [
        {
            "Прямой аналог": "#2e86ab",
            "Специализированный инструмент": "#f18f01",
            "Смежное решение": "#6a994e",
        }[item["class"]]
        for item in ordered
    ]

    fig, ax = plt.subplots(figsize=(10.0, 5.4))
    y = np.arange(len(labels))
    bars = ax.barh(
        y,
        values,
        color=colors,
        height=0.64,
        edgecolor="white",
        linewidth=0.6,
    )
    ax.set_yticks(y)
    ax.set_yticklabels(labels)
    ax.invert_yaxis()
    for bar, value in zip(bars, values):
        ax.text(
            value + 0.08,
            bar.get_y() + bar.get_height() / 2,
            f"{value:g}/10 ({value * 10:.0f}%)",
            va="center",
            fontsize=9,
            color="#222222",
        )
    ax.set_xlim(0, 11.4)
    ax.set_xlabel("Сумма подтверждённых признаков (из 10)", fontsize=9)
    ax.grid(axis="x", color="#e6e6e6", linewidth=0.8)
    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    handles = [
        Patch(facecolor="#2e86ab", label="Прямой аналог"),
        Patch(facecolor="#f18f01", label="Специализированный"),
        Patch(facecolor="#6a994e", label="Смежное решение"),
    ]
    ax.legend(handles=handles, loc="lower right", frameon=False, fontsize=8.5)
    finish(
        fig,
        ax,
        FIG_DIR / "15_analog_coverage.png",
        "Индекс покрытия функций аналогов",
        "Описательная сумма 10 признаков, а не оценка качества, зрелости или востребованности",
    )

figure_analog_coverage()



Если просто посчитать, сколько пунктов из моего списка закрывает инструмент, получается такой порядок:

- aptitude: 6 из 10;
- LDG: 5,5 из 10;
- pacwall: 4,5 из 10;
- mpm: 4 из 10;
- debtree: 3,5 из 10;
- apt-rdepends: 3 из 10;
- Graphite: 2,5 из 10;
- Starfish: 2 из 10.

Самое интересное начинается, если смотреть не на отдельные галочки, а на комбинации:

- граф и мульти-менеджерность есть только у LDG;
- граф и orphan-поиск есть только у pacwall, но только для pacman и без интерактива;
- мульти-менеджерность и orphan-поиск есть только у mpm, но без графа;
- orphan-поиск и предпросмотр удаления есть только у aptitude, но без графа;
- графа и предпросмотра удаления нет ни у кого одновременно;
- графа и моделирования установки тоже нет ни у кого одновременно.

Это говорит о наличии аналогов в каждом из сценариев, но не во всех сразу

Один старый кандидат, URPM, сейчас недоступен (репозиторий отдаёт 404), а потому он отсутствует в некоторых сравнениях.

## 7. Где может быть место DependenSee

DependenSee может казаться ещё одним проектом, решающим уже давно решённую задачу. Но на самом деле - отнюдь. Да, у проекта много конкурентов. Но "Нет ничего нового под солнцем". Но ни у одного из проектов не было возможности удалённого просмотра графа с удалённого сервера без предварительной загрузки файла, прямо в режиме реального времени. DependenSee не будет единственным. Зато он сможет собрать опыт конкурентов и добавить что-то от себя. Возможно, став лучшим решением для определённых задач.
## Итог

Корпус показывает три вещи.

Первое: основная проблема не в том, что пакетов много, а в том, что непонятно, какие из них нужны и что будет после удаления.
Второе: визуализация сама по себе интересна немногим, но запросы на неё очень точные. Люди хотят наглядности в управлении титаническими объёмами пакетов на linux-машинах.
Третье: аналоги закрывают отдельные куски задачи, но не весь спектр, а значит есть место для своей ниши.